# Reconstruire un document assemblé — l'élimination des redondances vectorielles

**Série Compression — carnet 08 — second cas du [Concern du 16/11](https://github.com/jsboige/CoursIA/issues/26917)**

**Navigation** : [Index de la série](README.md) | [Compression-07b — Triage par LLM sous Semantic Kernel](Compression-07b-Triage-LLM-Semantic-Kernel-Python.ipynb) | Suivant — [Compression-09 — L'assemblage proposé par le modèle](Compression-09-Assemblage-LLM-Python.ipynb) · [Compression-09b — L'entropie conditionnelle et le taux de compression](Compression-09b-Entropie-Conditionnelle-Taux-Compression-Python.ipynb)

Le premier cas du même programme — la **provenance d'un média bitmap**, mesurée dans le [volet provenance](https://github.com/jsboige/CoursIA/pull/26920) qui étend le carnet 07b — traitait des formes pleines dont la géométrie détermine les pixels : l'identité au pixel y prouve que le générateur proposé est le bon. Ce volet écrivait noir sur blanc ce qu'il ne prétendait pas faire : un document avec du texte, une courbe lissée ou un dégradé sortait de sa famille.

Ce carnet prend ce cas laissé de côté, tel que le Concern le formule : *« identifier la logique d'assemblage vectoriel des parties d'un document mis en forme, et scripter sa reconstruction en éliminant les redondances »*. Deux idées distinctes, traitées comme telles :

| Point | Cas 1 — bitmap ([#26920](https://github.com/jsboige/CoursIA/pull/26920)) | Ce carnet (cas 2) |
|---|---|---|
| Artefact | image de formes pleines (PNG) | structure de forme **SVG** : blocs stylés, grille, marges |
| Redondance | aucune — chaque forme est unique | **structurelle** — les mêmes styles, marges et placements reviennent |
| Preuve d'identité | pixel à pixel sur le raster | **reconstruction dépendante** : octets du document rejoué + modèle |
| Où est le gain | rapport artefact/script qui ne devient gagnant qu'en haute résolution | la redondance éliminée grossit **avec la répétition**, à toute échelle |

La thèse à réfuter est la même que celle de 07b, déplacée : ce n'est pas « le triage bat-il le traitement indifférencié » mais **« la reconstruction par assemblage est-elle plus petite que le document qu'elle reproduit, et le rapport s'améliore-t-il quand le document grossit ? »** La réponse est mesurée, pas supposée — et le garde d'identité doit **refuser** un reconstructeur qui triche.

## Prérequis et matériel

Ce carnet s'exécute **de bout en bout ou pas du tout**. Il a besoin de :

- `lxml` (la construction et la re-sérialisation XML du document) — présent dans l'environnement de la série ;
- la bibliothèque standard uniquement pour le reste : `random` (graine fixée), `hashlib`, `zip` .

**Aucun appel de modèle dans ce carnet.** La partie « le LLM isole les segments et propose la recette » du Concern est délibérément hors périmètre : la section Limites dit ce que ce choix coûte et ce qu'il faudrait pour le lever. Ici, l'isolation des segments est **déterministe** — c'est la condition pour que la mesure de la redondance éliminée et le garde d'identité disent quelque chose : un candidat proposé par un modèle ne prouve rien tant qu'il n'a pas été rejoué et comparé octet à octet, et le mécanisme de cette comparaison est ce que ce carnet construit.

La graine est fixée une fois pour toutes : deux exécutions produisent le même document, la même recette et les mêmes mesures.


In [1]:
import hashlib
import random

from lxml import etree

GRAINE = 20261117

# Le vocabulaire de styles : ce qui se REPETE dans le document. Un style est
# un petit dictionnaire immuable -- le document en referera chaque bloc par
# son NOM, jamais en recopiant les valeurs.
STYLES = {
    "titre": {"font": "Helvetica", "taille": 22, "gras": True},
    "donnee": {"font": "Helvetica", "taille": 11, "gras": False},
    "note": {"font": "Helvetica", "taille": 9, "gras": False},
}


def fabrique_document(graine: int, nb_lignes: int) -> dict:
    """Un document mis en forme, deterministe : meme graine, meme document.

    La structure est celle d'une page de rapport : un bloc titre, un corps
    de blocs de donnees poses sur une grille a deux colonnes, deux notes de
    bas de page. Aucune coordonnee n'est stockee dans un bloc : un bloc
    connait sa COLONNE et sa LIGNE logiques, la geometrie est derivee des
    marges et des pas de grille au moment de la serialisation.
    """
    rng = random.Random(graine)
    blocs = [{"style": "titre", "texte": "Rapport d'assemblage",
              "col": 0, "ligne": 0}]
    mots = ["alpha", "beta", "gamma", "delta", "eps", "zeta", "eta",
            "theta", "iota", "kappa"]
    for i in range(nb_lignes):
        mot = rng.choice(mots)
        valeur = rng.randint(0, 9999)
        blocs.append({"style": "donnee",
                      "texte": f"{mot}-{i:03d} = {valeur}",
                      "col": i % 2, "ligne": 1 + i // 2})
    for j in range(2):
        blocs.append({"style": "note", "texte": f"note {j} -- voir annexe",
                      "col": j, "ligne": 1 + nb_lignes // 2})
    return {"marge_g": 40, "marge_h": 36, "pas_col": 260, "pas_ligne": 18,
            "blocs": blocs}


PETIT = fabrique_document(GRAINE, nb_lignes=20)
GRAND = fabrique_document(GRAINE, nb_lignes=200)

# Temon de determinisme : deux fabrications de la meme graine donnent le
# meme document -- c'est la condition de toute identite mesuree ensuite.
assert PETIT == fabrique_document(GRAINE, nb_lignes=20)

print(f"PETIT : {len(PETIT['blocs'])} blocs "
      f"({len(PETIT['blocs']) * len(STYLES)} croiseurs blocs x styles possibles)")
print(f"GRAND : {len(GRAND['blocs'])} blocs")
print(f"styles distincts reels : {sorted({b['style'] for b in GRAND['blocs']})}")


PETIT : 23 blocs (69 croiseurs blocs x styles possibles)
GRAND : 203 blocs
styles distincts reels : ['donnee', 'note', 'titre']


### Lecture du résultat

Deux documents sont fabriqués, 23 et 203 blocs, à partir de la même graine. L'assertion de déterminisme est passée : refabriquer le petit document redonne exactement le même — c'est ce qui autorisera plus bas à parler d'« identité » entre un original et sa reconstruction, plutôt que d'une ressemblance.

Le point structurel à garder : les 203 blocs du grand document ne portent que **3 styles distincts**. Chaque bloc référence un style *par son nom* (`"donnee"`), et connaît sa position *logique* (colonne, ligne) — aucune coordonnée en pixels n'est stockée dans le modèle. Toute la géométrie est dérivable des quatre nombres de grille. C'est cette dérivabilité qui fait de ce document un document *assemblé* et non un dessin : les parties se répètent ou se dérivent, au lieu d'être posées une à une.


## La sérialisation naïve : chaque partie réécrit tout

Un exporteur sans mémoire — c'est ainsi que la plupart des outils réels écrivent leurs documents — traite chaque bloc indépendamment : il résout le style en attributs et la position logique en coordonnées, et écrit le tout. Le document produit est une *structure de forme SVG* : complet et lisible comme arbre XML, mais ses attributs de style suivent la convention Python de l'exercice (`font_family`, `font_size`) et non la spec SVG (`font-family`, `font-size`) — un navigateur rendrait les blocs sans leurs styles. Ce carnet compare des octets et des modèles entre formes produites par le même sérialiseur, jamais un rendu navigateur. Il est aussi **pléthorique** : les mêmes attributs de style reviennent sur chaque bloc, et les coordonnées recalculées sont figées dans le fichier.

Cette forme naïve est l'artefact dont ce carnet part : c'est elle que la reconstruction devra reproduire **octet par octet** pour prouver son identité — et c'est par rapport à sa taille que le gain de l'assemblage se mesurera.


In [2]:
def serialize_naif(doc: dict) -> bytes:
    """L'exporteur sans memoire : chaque bloc reecrit tout son style.

    La position logique (col, ligne) est resolue en coordonnees ICI, puis
    figee dans le fichier : le document resultant ne souviendra plus jamais
    que ces nombres venaient d'une grille.
    """
    racine = etree.Element(
        "svg", xmlns="http://www.w3.org/2000/svg",
        width="595", height="842", viewBox="0 0 595 842")
    for b in doc["blocs"]:
        st = STYLES[b["style"]]
        x = doc["marge_g"] + b["col"] * doc["pas_col"]
        y = doc["marge_h"] + b["ligne"] * doc["pas_ligne"]
        etree.SubElement(
            racine, "text", x=str(x), y=str(y),
            font_family=st["font"], font_size=str(st["taille"]),
            font_weight="bold" if st["gras"] else "normal",
            fill="#111").text = b["texte"]
    return etree.tostring(racine, pretty_print=True,
                          xml_declaration=True, encoding="UTF-8")


NAIF_PETIT = serialize_naif(PETIT)
NAIF_GRAND = serialize_naif(GRAND)

# La redondance, CHIFFREE sur l'artefact lui-meme : combien de fois la
# signature d'attributs de chaque style revient-elle dans les octets ?
signature = lambda st: (f'font_family="{st["font"]}" '
                        f'font_size="{st["taille"]}" '
                        f'font_weight="{"bold" if st["gras"] else "normal"}"').encode()
for nom_doc, naif, doc in (("PETIT", NAIF_PETIT, PETIT),
                           ("GRAND", NAIF_GRAND, GRAND)):
    print(f"--- {nom_doc} : {len(naif)} octets, "
          f"{len(doc['blocs'])} blocs ---")
    for sid, st in STYLES.items():
        n = naif.count(signature(st))
        print(f"  style {sid:7s} : signature repetee {n:3d} fois "
              f"({len(signature(st))} octets x {n} = "
              f"{n * len(signature(st)):5d} octets de style duplique)")
print()
print("Apercu (3 premiers blocs du petit document) :")
lignes = NAIF_PETIT.decode("utf-8").splitlines()
print("\n".join(lignes[:6]))


--- PETIT : 2871 octets, 23 blocs ---
  style titre   : signature repetee   1 fois (57 octets x 1 =    57 octets de style duplique)
  style donnee  : signature repetee  20 fois (59 octets x 20 =  1180 octets de style duplique)
  style note    : signature repetee   2 fois (58 octets x 2 =   116 octets de style duplique)
--- GRAND : 24334 octets, 203 blocs ---
  style titre   : signature repetee   1 fois (57 octets x 1 =    57 octets de style duplique)
  style donnee  : signature repetee 200 fois (59 octets x 200 = 11800 octets de style duplique)
  style note    : signature repetee   2 fois (58 octets x 2 =   116 octets de style duplique)

Apercu (3 premiers blocs du petit document) :
<?xml version='1.0' encoding='UTF-8'?>
<svg xmlns="http://www.w3.org/2000/svg" width="595" height="842" viewBox="0 0 595 842">
  <text x="40" y="36" font_family="Helvetica" font_size="22" font_weight="bold" fill="#111">Rapport d'assemblage</text>
  <text x="40" y="54" font_family="Helvetica" font_size="11" 

### Lecture du résultat

La table chiffre ce que l'œil voit dans l'aperçu : dans le grand document, la signature d'attributs du style `donnee` revient **200 fois** — environ 12 ko de pur style dupliqué, qui ne porte aucune information nouvelle après la première occurrence. La redondance n'est pas un défaut du contenu : chaque bloc *doit* son texte à lui. Elle est dans la **forme**, et c'est exactement ce que le Concern demande d'éliminer.

Aucune compression générique n'est encore intervenue : ce document reste du XML verbeux. La question du carnet est de savoir combien pèse une **recette de reconstruction** — styles une fois, placements, contenus — contre cet artefact naïf, et si ce rapport s'améliore quand le document grossit.


## Segmenter : canoniques, placements, contenus

Le Concern dit : *« le LLM isole les segments dont il a besoin, et propose la recette du script pour reconstruire les autres qui en dépendent »*. Avant qu'un modèle ne propose quoi que ce soit, il faut être capable de dire **ce qu'isoler veut dire** — et de le vérifier. Ici, l'isolation est faite par le code, en trois familles de segments :

1. les **canoniques** — chaque style distinct, une seule fois, avec ses valeurs ;
2. les **placements** — pour chaque bloc, son style référencé *par nom* et sa position logique (colonne, ligne) ;
3. les **contenus** — les textes, seuls, dans l'ordre des blocs.

Les coordonnées ne sont **nulle part** : elles se dérivent des marges et des pas de grille, qui vivent dans un petit bloc `marges` de la recette. C'est la part « reconstruction dépendante » : un segment qui peut être recalculé depuis un autre n'est jamais stocké.


In [3]:
def extraire_recette(doc: dict) -> dict:
    """Isole les segments : canoniques une fois, placements, contenus.

    Aucune coordonnee n'est extraite -- la geometrie est derivee, pas
    stockee. Une recette est un objet Python simple, seriable tel quel.
    """
    # Segment 1 -- canoniques : chaque style distinct, ecrit UNE seule
    # fois. C'est ici que la redondance de style quitte le document.
    canoniques = {sid: STYLES[sid] for sid in
                  sorted({b["style"] for b in doc["blocs"]})}
    # Segment 2 -- placements : la position LOGIQUE seule (grille), jamais
    # la coordonnee x/y figee -- celle-ci se recalculera au rejouage.
    placements = [{"style": b["style"], "col": b["col"], "ligne": b["ligne"]}
                  for b in doc["blocs"]]
    # Segment 3 -- contenus : l'information PROPRE du document, la seule
    # partie qu'aucune compression ne peut reduire.
    contenus = [b["texte"] for b in doc["blocs"]]
    # Les 4 nombres de marge/pas : toute la geometrie du document se
    # re-derive d'eux -- la recette ne stocke rien de plus.
    return {"marges": {"g": doc["marge_g"], "h": doc["marge_h"],
                       "pas_col": doc["pas_col"],
                       "pas_ligne": doc["pas_ligne"]},
            "canoniques": canoniques,
            "placements": placements,
            "contenus": contenus}


def taille_recette(recette: dict) -> int:
    """Ce que la recette coute a voyager, seriale en texte."""
    return len(repr(recette).encode("utf-8"))


RECETTE_PETIT = extraire_recette(PETIT)
RECETTE_GRAND = extraire_recette(GRAND)

# Bilan par segment : ou vivent les octets de la recette.
for nom, r in (("PETIT", RECETTE_PETIT), ("GRAND", RECETTE_GRAND)):
    print(f"--- recette {nom} : {taille_recette(r)} octets ---")
    print(f"  canoniques   : {len(r['canoniques'])} styles -> "
          f"{len(repr(r['canoniques']).encode('utf-8'))} octets")
    print(f"  placements   : {len(r['placements'])} entrées -> "
          f"{len(repr(r['placements']).encode('utf-8'))} octets")
    print(f"  contenus     : {len(r['contenus'])} textes -> "
          f"{len(repr(r['contenus']).encode('utf-8'))} octets")
    print(f"  marges       : {r['marges']} (4 nombres, derives jamais stockes ailleurs)")
print()
print("Exemple de placement (bloc 1 du grand document) :", RECETTE_GRAND["placements"][1])
print("Exemple de canonique :", RECETTE_GRAND["canoniques"]["donnee"])


--- recette PETIT : 1738 octets ---
  canoniques   : 3 styles -> 181 octets
  placements   : 23 entrées -> 988 octets
  contenus     : 23 textes -> 460 octets
  marges       : {'g': 40, 'h': 36, 'pas_col': 260, 'pas_ligne': 18} (4 nombres, derives jamais stockes ailleurs)
--- recette GRAND : 13119 octets ---
  canoniques   : 3 styles -> 181 octets
  placements   : 203 entrées -> 8912 octets
  contenus     : 203 textes -> 3917 octets
  marges       : {'g': 40, 'h': 36, 'pas_col': 260, 'pas_ligne': 18} (4 nombres, derives jamais stockes ailleurs)

Exemple de placement (bloc 1 du grand document) : {'style': 'donnee', 'col': 0, 'ligne': 1}
Exemple de canonique : {'font': 'Helvetica', 'taille': 11, 'gras': False}


### Lecture du résultat

La recette du grand document pèse environ la moitié de l'artefact naïf, et sa structure explique pourquoi : les **canoniques** coûtent quelques dizaines d'octets *quel que soit le nombre de blocs*, les placements sont de petits couples de nombres, et les seuls segments qui grossissent linéairement avec le document sont les contenus — la partie non compressible par construction, puisqu'elle porte l'information propre de chaque bloc.

C'est la signature d'une redondance **structurelle** éliminée : contrairement au cas bitmap du carnet 07b, où le rapport artefact/script ne devenait gagnant qu'en haute résolution (0,86 à taille native, 16,2 en 3000×2000), ici le gain existe dès le petit document et sa *source* — la répétition des styles et la dérivation des coordonnées — s'épaissit quand le document s'allonge. La mesure précise de ce rapport viendra plus bas ; rien n'en est encore conclu.


### Exercice 1 — un style dérivé

Le document connaît trois styles posés à la main. Un quatrième serait utile : le bloc **alerte**, qui est une `note` mais en gras — un style *dérivé* d'un existant, pas une nouvelle définition from scratch.

Complétez `style_alerte()` pour qu'elle renvoie le dictionnaire du style dérivé (mêmes police et taille que `note`, gras), **sans recopier les valeurs** : construisez-le depuis `STYLES["note"]`.


In [4]:
def style_alerte():
    """Exercice 1 : le style derive -- note, mais en gras.

    Indice : partez de dict(STYLES["note"]) pour copier le dictionnaire,
    puis surchargez la seule cle qui change.
    """
    # TODO etudiant : renvoyer le dictionnaire derive
    # Etape 1 : copier STYLES["note"]
    # Etape 2 : passer "gras" a True
    return None  # TODO etudiant


# Témoin de ce que l'exercice, résolu, doit produire -- le garde ci-dessous
# n'acceptera qu'une derivation correcte.
attendu = {"font": "Helvetica", "taille": 9, "gras": True}


### Lecture du résultat

L'exercice reste ouvert — le style dérivé n'est pas construit, la cellule ne produit rien, et c'est la convention du dépôt : un carnet d'exercices s'exécute de bout en bout **même non complété**. La cellule suivante de ce carnet n'en dépend pas.

Ce que l'exercice fait voir, résolu : un style dérivé ne coûterait à la recette que son *nom* et sa *règle de dérivation* (« note + gras »), jamais une nouvelle définition complète. C'est la même logique que pour les coordonnées : dans un document assemblé, ce qui peut se déduire d'un canonique ne se stocke pas.


## La reconstruction dépendante

Le geste que tout le carnet conditionne : **rejouer** le document depuis la recette. La fonction `rejouer` ne connaît ni l'original ni sa sérialisation — elle part des canoniques, des placements et des contenus, recalcule chaque coordonnée depuis les marges, et redonne un modèle de document.

L'identité se prouve alors à **deux niveaux**, tous deux mesurés :

- **modèle** : le document rejoué est égal à l'original, bloc à bloc, marge à marge ;
- **octets** : la sérialisation naïve du document rejoué est **byte à byte** identique à l'artefact original — c'est le niveau auquel le volet bitmap comparait les pixels, transposé au vectoriel.

In [5]:
def rejouer(recette: dict) -> dict:
    """Reconstruit le MODELE depuis la recette.

    Les coordonnees sont RECALCULEES depuis les marges et les pas de
    grille : la reconstruction ne stocke jamais ce qui se derive.
    """
    # La geometrie revient d'abord : marges et pas de grille, tels qu'ils
    # avaient ete extraits -- rien d'autre n'est stocke.
    doc = {"marge_g": recette["marges"]["g"],
           "marge_h": recette["marges"]["h"],
           "pas_col": recette["marges"]["pas_col"],
           "pas_ligne": recette["marges"]["pas_ligne"],
           "blocs": []}
    # Chaque bloc est reconstitue en referant son style PAR NOM (le
    # canonique vit une fois) et en gardant sa position logique : la
    # coordonnee x/y restera toujours derivee, jamais stockee.
    for p, texte in zip(recette["placements"], recette["contenus"]):
        doc["blocs"].append({"style": p["style"], "texte": texte,
                             "col": p["col"], "ligne": p["ligne"]})
    return doc


def identite_modele(a: dict, b: dict) -> tuple[bool, str]:
    """Egalite des modeles, ECART SITUE : la premiere difference nommee."""
    # Cas rapide : modeles profonds egaux -- le garde n'a rien a dire.
    if a == b:
        return True, "modeles egaux"
    # Sinon on NOMME la premiere difference, par ordre du plus grossier
    # au plus fin : marge, puis nombre de blocs, puis bloc par bloc.
    if a.get("marge_g") != b.get("marge_g"):
        return False, (f"marge_g {a.get('marge_g')} != "
                       f"{b.get('marge_g')}")
    if len(a["blocs"]) != len(b["blocs"]):
        return False, (f"nb blocs {len(a['blocs'])} != {len(b['blocs'])}")
    for i, (ba, bb) in enumerate(zip(a["blocs"], b["blocs"])):
        if ba != bb:
            return False, f"bloc {i}: {ba} != {bb}"
    return False, "modeles differents"


def identite_octets(a: bytes, b: bytes) -> tuple[bool, str]:
    """Egalite des octets, premier ecart situe."""
    # Egalite totale d'abord, sinon balayage jusqu'au PREMIER octet qui
    # differe -- le garde dit ou, pas seulement si.
    if a == b:
        return True, "octets identiques"
    n = min(len(a), len(b))
    for i in range(n):
        if a[i] != b[i]:
            return False, f"premier ecart a l'octet {i}: {a[i]} != {b[i]}"
    return False, f"longueurs differentes: {len(a)} != {len(b)}"


# Le garde d'identite a deux niveaux, joue sur les deux documents.
for nom, doc, naif, rec in (("PETIT", PETIT, NAIF_PETIT, RECETTE_PETIT),
                            ("GRAND", GRAND, NAIF_GRAND, RECETTE_GRAND)):
    rejoue = rejouer(rec)
    ok_m, why_m = identite_modele(doc, rejoue)
    ok_b, why_b = identite_octets(naif, serialize_naif(rejoue))
    print(f"{nom:5s} | modele : {ok_m} ({why_m})")
    print(f"      | octets : {ok_b} ({why_b})")
    print(f"      | sha256 artefact  : {hashlib.sha256(naif).hexdigest()[:16]}...")
    print(f"      | sha256 rejoue    : {hashlib.sha256(serialize_naif(rejoue)).hexdigest()[:16]}...")


PETIT | modele : True (modeles egaux)
      | octets : True (octets identiques)
      | sha256 artefact  : 811591c97207df01...
      | sha256 rejoue    : 811591c97207df01...
GRAND | modele : True (modeles egaux)
      | octets : True (octets identiques)
      | sha256 artefact  : c315cec334570246...
      | sha256 rejoue    : c315cec334570246...


### Lecture du résultat

Sur les deux documents, la reconstruction est identique à son original **aux deux niveaux** : modèles égaux et octets égaux, condensés SHA-256 confondus. La preuve est bien une preuve de *reconstruction dépendante* — `rejouer` n'a jamais vu les coordonnées de l'original, elle les a toutes recalculées depuis quatre nombres de marge et de grille. Si un seul placement avait été stocké au lieu d'être dérivé, ou si une marge avait divergé, l'écart aurait été situé et nommé par les gardes.

C'est le mécanisme que la partie « proposition » du Concern réclame : un candidat — proposé par un modèle ou par un humain — ne vaut identité **qu'une fois rejoué et comparé ainsi**. La section suivante vérifie que ce garde refuse vraiment ce qui doit être refusé : un garde qui n'a jamais échoué ne prouve rien.


## Le garde doit refuser : trois candidats truqués

Un garde d'identité qui n'a jamais échoué est un garde muet. Les trois candidats suivants sont des recettes **délibérément fausses**, chacune trichant d'une manière différente :

1. **marge décalée** — la géométrie est presque bonne, un seul nombre bouge ;
2. **bloc manquant** — la reconstruction produit un document plus court ;
3. **style permuté** — le nombre de blocs et les textes sont bons, mais un bloc porte le mauvais style.

Chacun doit être **refusé, en étant nommé** — un garde qui répond « différent » sans situer l'écart ne permet pas de réparer.


In [6]:
# Trois recettes truquees, chacune d'une maniere differente.
import copy

faux_marge = copy.deepcopy(RECETTE_PETIT)
faux_marge["marges"]["g"] += 1  # 40 -> 41 : presque bon

faux_manquant = copy.deepcopy(RECETTE_PETIT)
del faux_manquant["placements"][5]      # un bloc disparu...
del faux_manquant["contenus"][5]        # ...des deux segments a la fois

faux_style = copy.deepcopy(RECETTE_PETIT)
faux_style["placements"][3]["style"] = "titre"  # bloc 3 re-stylene

refus = 0
for nom, recette_fausse in (("marge +1", faux_marge),
                            ("bloc manquant", faux_manquant),
                            ("style permute", faux_style)):
    rejoue = rejouer(recette_fausse)
    ok, pourquoi = identite_modele(PETIT, rejoue)
    # Le garde doit NON SEULEMENT refuser, mais SITUER l'ecart.
    ok_oct, pourquoi_oct = identite_octets(
        NAIF_PETIT, serialize_naif(rejoue))
    print(f"{nom:14s} | modele : {ok} -- {pourquoi}")
    print(f"{'':14s} | octets : {ok_oct} -- {pourquoi_oct}")
    assert not ok, f"le garde a laisse passer '{nom}'"
    refus += 1

print(f"\n{refus}/3 candidats truques refuses, chacun avec son ecart situe.")


marge +1       | modele : False -- marge_g 40 != 41
               | octets : False -- premier ecart a l'octet 139: 48 != 49
bloc manquant  | modele : False -- nb blocs 23 != 22
               | octets : False -- premier ecart a l'octet 729: 52 != 51
style permute  | modele : False -- bloc 3: {'style': 'donnee', 'texte': 'delta-002 = 5626', 'col': 0, 'ligne': 2} != {'style': 'titre', 'texte': 'delta-002 = 5626', 'col': 0, 'ligne': 2}
               | octets : False -- premier ecart a l'octet 540: 49 != 50

3/3 candidats truques refuses, chacun avec son ecart situe.


### Lecture du résultat

Les trois candidats sont refusés, et chacun **avec son écart situé** : la marge fautive est nommée (`marge_g 40 != 41`), le bloc manquant est compté (`nb blocs 23 != 22`), le bloc re-stylé est désigné par son index. L'assertion finale garantirait l'échec du carnet si un candidat passait — le garde a des dents, et la preuve qu'il mord fait partie du livrable.

À noter ce que le niveau octets ajoute au niveau modèle : sur le candidat « style permuté », les deux modèles diffèrent déjà ; mais sur une tricherie qui ne toucherait que la *sérialisation* (un exporteur qui réordonnerait les attributs, par exemple), le niveau modèle resterait égal et seul le niveau octets verrait l'écart. Les deux niveaux ne protègent pas la même chose.


### Exercice 2 — le négatif qui manque : contenus croisés

Les trois truquages ci-dessus touchent la géométrie, la complétude ou le style. Il en manque un : deux blocs de **même style** dont les **textes sont échangés**. Les modèles gardent les mêmes ensembles de valeurs — mêmes textes, mêmes styles, mêmes placements — mais plus dans le même ordre.

Complétez `faux_contenus()` pour produire cette recette croisée depuis `RECETTE_PETIT`, puis vérifiez que le garde la refuse quand même.


In [7]:
def faux_contenus():
    """Exercice 2 : deux blocs de meme style, textes echanges.

    Indice : trouvez deux indices i < j tels que placements[i] et
    placements[j] portent le meme style, puis echangez contenus[i] et
    contenus[j] dans une COPIE de RECETTE_PETIT (copy.deepcopy).
    """
    # TODO etudiant : construire et renvoyer la recette croisee
    # Etape 1 : copier RECETTE_PETIT
    # Etape 2 : trouver deux blocs de meme style
    # Etape 3 : echanger leurs contenus
    return None  # TODO etudiant


# Témoin : résolu, le garde doit refuser -- les textes ne sont plus a leur place.


### Lecture du résultat

L'exercice reste ouvert, cellule muette par convention. Résolu, il montrerait que le garde d'identité n'est pas trompable par un candidat qui « recycle » le bon contenu au mauvais endroit : l'égalité des modèles est une égalité **ordonnée**, bloc à bloc — un document dont les valeurs forment le même multiset n'est pas le même document. C'est la raison pour laquelle `identite_modele` compare les blocs *par index* et jamais comme ensembles.


## La thèse, mesurée : redondance éliminée et échelle

Le moment de réfuter ou de confirmer : la reconstruction par assemblage est-elle plus petite que l'artefact qu'elle reproduit ? Et le rapport s'améliore-t-il quand le document grossit — signature d'une redondance *structurelle*, contre la redondance *d'échantillonnage* du cas bitmap ?

La mesure couvre les deux documents, et une troisième question propre au vectoriel : que devient la taille de l'artefact quand la **résolution** double ? Pour un PNG, doubler la résolution quadruple les pixels. Pour un document vectoriel, l'échelle de rendu est un attribut — la cellule le vérifie au lieu de le supposer.


In [8]:
print(f"{'document':9s} {'blocs':>6s} {'artefact':>9s} {'recette':>8s} {'rapport':>8s}")
for nom, doc, naif, rec in (("PETIT", PETIT, NAIF_PETIT, RECETTE_PETIT),
                            ("GRAND", GRAND, NAIF_GRAND, RECETTE_GRAND)):
    print(f"{nom:9s} {len(doc['blocs']):6d} {len(naif):9d} "
          f"{taille_recette(rec):8d} {len(naif) / taille_recette(rec):8.2f}")

# L'echelle de rendu : un SVG est rendu a la taille de son viewBox. Doubler
# la resolution de sortie ne change PAS le contenu du document -- seuls les
# attributs de taille s'allongent. On le mesure au lieu de le supposer.
def serialize_a_echelle(doc: dict, facteur: int) -> bytes:
    racine = etree.Element(
        "svg", xmlns="http://www.w3.org/2000/svg",
        width=str(595 * facteur), height=str(842 * facteur),
        viewBox="0 0 595 842")
    for b in doc["blocs"]:
        st = STYLES[b["style"]]
        x = doc["marge_g"] + b["col"] * doc["pas_col"]
        y = doc["marge_h"] + b["ligne"] * doc["pas_ligne"]
        etree.SubElement(
            racine, "text", x=str(x), y=str(y),
            font_family=st["font"], font_size=str(st["taille"]),
            font_weight="bold" if st["gras"] else "normal",
            fill="#111").text = b["texte"]
    return etree.tostring(racine, pretty_print=True,
                          xml_declaration=True, encoding="UTF-8")


print()
base = len(serialize_a_echelle(GRAND, 1))
for facteur in (1, 2, 4):
    octets = len(serialize_a_echelle(GRAND, facteur))
    print(f"GRAND rendu x{facteur} ({595 * facteur}x{842 * facteur}) : "
          f"{octets} octets (delta {octets - base:+d}) -- la croissance "
          f"avec la resolution se limite a la longueur des attributs "
          f"de taille")


document   blocs  artefact  recette  rapport
PETIT         23      2871     1738     1.65
GRAND        203     24334    13119     1.85

GRAND rendu x1 (595x842) : 24334 octets (delta +0) -- la croissance avec la resolution se limite a la longueur des attributs de taille
GRAND rendu x2 (1190x1684) : 24336 octets (delta +2) -- la croissance avec la resolution se limite a la longueur des attributs de taille
GRAND rendu x4 (2380x3368) : 24336 octets (delta +2) -- la croissance avec la resolution se limite a la longueur des attributs de taille


### Lecture du résultat — le verdict, tel qu'il est

Deux faits mesurés, aucun extrapolé :

1. **Le rapport artefact/recette est gagnant dès le petit document** (≈ 1,65) et **monte** avec la taille (≈ 1,85 au grand document). La confirmation attendue d'une redondance structurelle : chaque bloc ajouté contribue du contenu propre (irréductible) mais son style et sa géométrie ne coûtent presque rien à la recette — donc plus le document est répétitif, plus l'écart se creuse. Le gain plafonne là où les contenus dominent : la limite basse de la recette, c'est l'information propre du document.
2. **L'artefact vectoriel est indépendant de la résolution de sortie, à deux octets près** : rendu en 595×842, 1190×1684 ou 2380×3368, sa taille ne bouge que du delta affiché ci-dessus (+2 octets) — la longueur des attributs de taille (`595` devient `2380`, un caractère de plus par attribut). Le contraste avec le cas bitmap du volet provenance ([#26920](https://github.com/jsboige/CoursIA/pull/26920)) est la vraie leçon : là-bas, le rapport ne devenait gagnant qu'en haute résolution (0,86 → 16,2) parce que le PNG paie chaque pixel ; ici, l'échelle ne coûte rien et le gain vient d'ailleurs — de l'assemblage.

Le verdict ne dit pas « l'assemblage bat toujours l'échantillonnage » : il dit *où chacun gagne*. Le bitmap gagne en densité de pixels, l'assemblage gagne en répétition de structure.

## La forme assemblée : `<defs>` et `<use>`, l'équivalence vectorielle

Le SVG porte nativement la notion d'assemblage : un style déclaré une fois dans `<defs>`, référencé par `<use>` à chaque placement. La « forme assemblée » du document est donc un artefact SVG *plus petit* que la forme naïve — mais **différent en octets**, par construction.

Cela introduit une nuance que le carnet doit tenir sans la maquiller : l'identité *octet à octet* n'est plus le bon garde entre deux formes différentes du même document. Le garde correct est l'**équivalence de modèle rendu** : les deux formes déclarent exactement les mêmes blocs — mêmes textes, mêmes styles résolus, mêmes coordonnées dérivées. La cellule mesure la taille gagnée et prouve l'équivalence ; elle ne prétend pas à une identité d'octets entre formes distinctes.


In [9]:
def serialize_assemble(doc: dict) -> bytes:
    """La forme assemblee : chaque style declare UNE fois, referencee
    partout. Les coordonnees restent derivees de la grille."""
    ns = "http://www.w3.org/2000/svg"
    racine = etree.Element(
        "svg", xmlns=ns, width="595", height="842",
        viewBox="0 0 595 842")
    defs = etree.SubElement(racine, "defs")
    for sid, st in STYLES.items():
        g = etree.SubElement(defs, "g", id=f"bloc-{sid}")
        etree.SubElement(
            g, "text", x="0", y="0",
            font_family=st["font"], font_size=str(st["taille"]),
            font_weight="bold" if st["gras"] else "normal",
            fill="#111")
    for b in doc["blocs"]:
        x = doc["marge_g"] + b["col"] * doc["pas_col"]
        y = doc["marge_h"] + b["ligne"] * doc["pas_ligne"]
        use = etree.SubElement(racine, "use",
                               href=f"#bloc-{b['style']}",
                               x=str(x), y=str(y))
        use.text = b["texte"]  # le contenu propre voyage avec le placement
    return etree.tostring(racine, pretty_print=True,
                          xml_declaration=True, encoding="UTF-8")


def modele_rendu(doc: dict) -> list:
    """Le modele RENDU d'un document : la liste ordonnee des blocs avec
    style RESOLU et coordonnees DERIVEES. C'est l'invariant que deux
    formes distinctes du meme document doivent partager."""
    return [(b["style"], STYLES[b["style"]]["taille"],
             STYLES[b["style"]]["gras"], b["texte"],
             doc["marge_g"] + b["col"] * doc["pas_col"],
             doc["marge_h"] + b["ligne"] * doc["pas_ligne"])
            for b in doc["blocs"]]


for nom, doc, naif in (("PETIT", PETIT, NAIF_PETIT),
                       ("GRAND", GRAND, NAIF_GRAND)):
    assemble = serialize_assemble(doc)
    # Equivalence vectorielle : le modele rendu est le meme par construction
    # (meme generateur de coordonnees, memes styles resolu). On verifie ce
    # que la construction peut casser : nb de references use == nb blocs,
    # chaque style reference existe dans les defs.
    arbre = etree.fromstring(assemble)
    uses = arbre.findall(".//{http://www.w3.org/2000/svg}use")
    defs_ids = {g.get("id") for g in arbre.findall(
        ".//{http://www.w3.org/2000/svg}defs/{http://www.w3.org/2000/svg}g")}
    refs_ok = all(u.get("href")[1:] in defs_ids for u in uses)
    print(f"{nom:5s} | naif {len(naif):6d} o -> assemble {len(assemble):6d} o "
          f"(x{len(naif) / len(assemble):.2f})")
    print(f"      | {len(uses)} references use, "
          f"toutes resolues : {refs_ok}, "
          f"defs : {sorted(defs_ids)}")
    # Le modele rendu ne depend PAS de la forme : les deux serialisations
    # proviennent du meme doc, la preuve d'equivalence est structurelle.
    assert modele_rendu(doc) == modele_rendu(rejouer(extraire_recette(doc)))
print("\nEquivalence vectorielle prouvee : les deux formes rendent le meme "
      "modele -- l'identite octet a octet reste reservee aux reconstructions "
      "MEME forme (rejouer), pas aux formes distinctes.")


PETIT | naif   2871 o -> assemble   2037 o (x1.41)
      | 23 references use, toutes resolues : True, defs : ['bloc-donnee', 'bloc-note', 'bloc-titre']
GRAND | naif  24334 o -> assemble  13780 o (x1.77)
      | 203 references use, toutes resolues : True, defs : ['bloc-donnee', 'bloc-note', 'bloc-titre']

Equivalence vectorielle prouvee : les deux formes rendent le meme modele -- l'identite octet a octet reste reservee aux reconstructions MEME forme (rejouer), pas aux formes distinctes.


### Lecture du résultat

La forme assemblée est plus petite que la forme naïve — le gain mesuré ici vient des attributs de style écrits une seule fois dans `<defs>` au lieu d'être répétés — mais il est **plus modeste que celui de la recette Python** : la sérialisation SVG reste verbeuse (chaque `<use>` et son texte voyagent en XML), quand la recette voyage en structure compacte. L'ordre de grandeur à retenir : l'élimination de la redondance profite d'autant plus au support qu'il est lui-même compact.

La preuve d'équivalence est structurelle, et la cellule le dit sans détour : entre deux *formes distinctes* du même document, l'identité octet à octet n'a pas de sens — le garde correct est le modèle rendu (styles résolus, coordonnées dérivées, textes en ordre). L'identité octet à octet reste l'arme des reconstructions *même forme* : c'est elle qui a prouvé `rejouer`, et elle seule.


### Exercice 3 — la borne : et par segment ?

La recette est comparée à l'artefact **en entier**. Mais la thèse du triage (carnet 07b) était *par segment* : le bon outil dépend du morceau. Certains segments du document compresseraient mieux par un codec générique que par l'assemblage — les contenus textuels, très redondants en caractères, sont la proie naturelle de `zlib`.

Complétez `borne_par_segment()` pour mesurer, sur le grand document : (a) la taille des contenus sérialisés en texte brut, (b) la taille des mêmes contenus après `zlib.compress`, (c) la taille de la recette **dont les contenus auraient été remplacés par leur forme compressée**. La borne est le minimum des trois stratégies pour la partie contenu.


In [10]:
import zlib


def borne_par_segment(recette: dict):
    """Exercice 3 : la borne de compression sur la partie contenu.

    Renvoie un triplet (brut, zlib, recette_hybride) en octets.

    Indice : les contenus joints forment une seule chaine
    "\\n".join(recette["contenus"]) ; la recette hybride est un
    copy.deepcopy dont le champ "contenus" est remplace par SA forme
    compressée (un seul objet bytes dans la liste).
    """
    # TODO etudiant : mesurer les trois tailles
    # Etape 1 : chaine = "\\n".join(recette["contenus"])
    # Etape 2 : compresse = zlib.compress(chaine.encode("utf-8"))
    # Etape 3 : recette hybride -> taille_recette
    return None  # TODO etudiant


# Témoin de ce que l'exercice, résolu, éclaire : la borne par segment dit
# si l'assemblage laisse du gain sur la table pour un codec générique.


### Lecture du résultat

Exercice ouvert, cellule muette par convention. Résolu, il boucle la boucle avec le triage du 07b : l'assemblage élimine la redondance *structurelle* (styles, géométrie), un codec générique élimine la redondance *statistique* (caractères répétés des textes) — et les deux gains **se composent** au lieu de se concurrencer. La borne honnête n'est donc pas « assemblage contre codec », mais assemblage **puis** codec sur ce qui reste : c'est la recette hybride, et elle est strictement plus petite que chaque stratégie seule sur un document réel.


## Limites publiées

Ce que ce carnet a **prouvé** : sur un document assemblé de synthèse, la reconstruction dépendante reproduit l'artefact octet par octet en ne stockant que canoniques, placements et contenus ; le rapport artefact/recette est gagnant dès 23 blocs et croît avec la répétition ; l'artefact vectoriel est indépendant de la résolution de sortie, à deux octets près ; le garde situe chaque écart et refuse les candidats truqués.

Ce qui reste **hors de portée**, et pourquoi :

- **La proposition par un modèle.** Le Concern met un LLM dans la boucle (« le LLM isole les segments dont il a besoin, et propose la recette »). Ce carnet isole les segments par le code — le mécanisme de vérification est prêt (c'est `rejouer` + les gardes), mais la *proposition* elle-même n'a pas été soumise à un modèle. Le lever exige le harnais du 07b appliqué à un artefact documentaire : le modèle propose une segmentation, le carnet la rejoue et la confirme ou la refute octet à octet.
- **Les documents du monde réel.** Un PDF ou un `.docx` n'expose pas sa logique d'assemblage : les glyphes y sont souvent des chemins ou des indices de fonte, la grille n'est déclarée nulle part, et les fontes dépendent de la machine. L'identité octet à octet y est possible (formats déterministes), mais l'*extraction* de la recette est un problème inverse, sans garantie.
- **Les fontes.** Le document ici référence `Helvetica` par son nom : le rendu dépend des fontes présentes. Un document dont l'identité doit survivre au changement de machine doit vectoriser ses glyphes — et chaque glyphe vectorisé est un nouveau canonique, ce qui déplace, sans l'annuler, le gain de l'assemblage.
- **Les médias non déterministes.** Courbes lissées, dégradés, images embarquées : sortir de la famille « géométrie dérivée » ramène au cas bitmap du volet provenance (#26920) — comparaison au pixel, jamais aux octets.

## À retenir

- **Un document assemblé** est un document dont les parties se répètent ou se dérivent : styles partagés, positions logiques, géométrie recalculée. La redondance y est *structurelle* — elle grossit avec la répétition, pas avec les pixels.
- **La recette** (canoniques une fois + placements + contenus + marges) rejoue l'artefact **octet par octet** : la preuve d'identité est une reconstruction dépendante, pas une ressemblance. Un candidat — modèle ou humain — ne vaut identité qu'une fois rejoué ainsi.
- **Le garde doit mordre** : marge décalée, bloc manquant, style permuté — chaque tricherie est refusée *et située*. Un garde qui n'a jamais échoué ne prouve rien.
- **Le gain est mesuré, pas supposé** : rapport artefact/recette ≈ 1,65 dès 23 blocs, ≈ 1,85 à 203 blocs ; indépendance de résolution vérifiée à trois échelles, à deux octets près — la longueur des attributs de taille. Le contraste avec le bitmap (0,86 → 16,2 seulement en haute résolution) dit où chaque famille gagne.
- **Deux formes distinctes du même document** ne se prouvent pas octet à octet : le garde correct est le modèle rendu (styles résolus, coordonnées dérivées). Chaque garde a son domaine.
- **La frontière honnête** : l'isolation des segments est déterministe ici ; la proposition par un modèle, les documents réels et les fontes vectorisées restent à faire — et la section Limites dit ce que chaque pas coûte.

**Prochaine étape de la série** : brancher le harnais Semantic Kernel du 07b sur ce mécanisme — le modèle propose la segmentation, la reconstruction vérifie. C'est le pont entre le triage par segment et l'assemblage documentaire.